# Jedan SMS primer i jedan PyTorch batch

Do sada smo imali Python liste ID-jeva i maske. `SMSDataset` jednu poruku i njenu oznaku pretvara u tri PyTorch tenzora. `DataLoader` uzima više takvih primera i slaže ih u batch. Ovde još ne pravimo niti treniramo Transformer.

## Mali primer sa dve poruke

Oznaka klase je druga vrsta broja od ID-ja tokena: `ham = 0`, `spam = 1`. Rečnik tokena gradimo samo iz poruka koje nam služe za učenje.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import torch
from torch.utils.data import DataLoader

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.sms_dataset import LABEL_TO_ID, SMSDataset
from src.vocabulary import build_vocabulary

toy = pd.DataFrame({"label": ["ham", "spam"], "text": ["See you soon", "Free prize now!"]})
toy_vocabulary = build_vocabulary(toy["text"])
toy_dataset = SMSDataset(toy, toy_vocabulary, max_length=8)
example = toy_dataset[1]
print("Mapiranje oznaka:", LABEL_TO_ID)
pd.DataFrame({
    "polje": list(example),
    "tip": [str(value.dtype) for value in example.values()],
    "vrednost": [value.tolist() for value in example.values()],
})

Mapiranje oznaka: {'ham': 0, 'spam': 1}


,polje,tip,vrednost
0,input_ids,torch.int64,"[3, 5, 4, 2, 0, 0, 0, 0]"
1,padding_mask,torch.bool,"[False, False, False, False, True, True, True,..."
2,label,torch.int64,1


U jednom primeru `input_ids` i `padding_mask` imaju po 8 vrednosti. `label` je jedan broj. Maska je tipa `torch.bool`; ID-jevi i oznaka su celobrojni tenzori.

In [2]:
toy_loader = DataLoader(toy_dataset, batch_size=2, shuffle=False)
toy_batch = next(iter(toy_loader))
pd.DataFrame({
    "polje": list(toy_batch),
    "oblik": [str(tuple(value.shape)) for value in toy_batch.values()],
    "tip": [str(value.dtype) for value in toy_batch.values()],
})

,polje,oblik,tip
0,input_ids,"(2, 8)",torch.int64
1,padding_mask,"(2, 8)",torch.bool
2,label,"(2,)",torch.int64


Prva dimenzija `2` znači dve poruke u batch-u, druga dimenzija `8` znači osam pozicija po poruci. Oblik oznaka je `(2,)`: po jedna oznaka za svaku poruku. `shuffle=False` ovde olakšava pregled; tokom stvarnog treniranja redosled ćemo mešati.

## Prvi batch iz trening skupa

Ovde gradimo rečnik samo iz `sms_train.csv`, pa pravimo `SMSDataset` nad istim trening porukama. Izdvojeni test se ne čita.

In [3]:
train = pd.read_csv(project_root / "data" / "processed" / "sms_train.csv")
vocabulary = build_vocabulary(train["text"])
train_dataset = SMSDataset(train, vocabulary, max_length=128)
train_batch = next(iter(DataLoader(train_dataset, batch_size=4, shuffle=False)))
print("Broj poruka u trening Dataset-u:", len(train_dataset))
print("Oblici:", {name: tuple(value.shape) for name, value in train_batch.items()})
print("Oznake prvog batch-a:", train_batch["label"].tolist())

Broj poruka u trening Dataset-u: 4127
Oblici: {'input_ids': (4, 128), 'padding_mask': (4, 128), 'label': (4,)}
Oznake prvog batch-a: [0, 0, 1, 0]


Sada je putanja `tekst → tokeni → ID-jevi → dopuna i maska → tenzori → batch` spremna. Naredna celina je sam model: embedding, pozicije, Transformer encoder, prosek stvarnih tokena i klasifikator. Nema još izlaza modela, funkcije gubitka niti treniranja.